# pyGeoMapper — Notebook 1: Spectral Indices, DEM Surface Change (DoD), and 64-Band AlphaEarth Embedding Change

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GeoLarryLai/pyGeoMapper/blob/main/pyGeoMapper_1_spectral_and_surface_change.ipynb)

This notebook demonstrates the core spectral and surface-change workflow of `pyGeoMapper` (Python interface + parallel Rust core `pygeomapper._core`) on a real, 100% public-domain study patch covering the **2014 Oso (SR 530) Landslide & North Fork Stillaguamish River valley, Washington State, USA** (`EPSG:32610`, UTM Zone 10N, `200 × 200` grid at ~10 m spacing):

1. **CRS-Aware Raster I/O & Terrain Morphometry (`raster_io`, `terrain_analysis`)**: Load pre-landslide (`Copernicus GLO-30`, 2011–2013) and post-landslide (`USGS 3DEP` airborne LiDAR) DEMs alongside bi-temporal Copernicus Sentinel-2 L2A surface reflectance cubes (`2017-08-26` vs. `2024-09-06`). Compute Horn (1981) slope, aspect, multi-azimuth hillshade, curvature, and Topographic Position Index (TPI) in Rust.
2. **Optical & Mineral Spectral Indices (`spectral_indices`)**: Compute `NDVI`, `NBR`, `NDWI`, `NDMI`, `NDBI`, `BSI` (Bare Soil Index), `ClayRatio` (`SWIR1/SWIR2`), `FerricOxideRatio`, and `FerrousIronIndex` in Rust.
3. **Bi-Temporal Surface & Elevation Change (`surface_change`)**: Compute the DEM of Difference (`DoD = z_post - z_pre`), planar bias detrending and Nuth & Kääb (2011) co-registration shift check, eroded/deposited volume budget ($\text{m}^3$), spectral change ($\Delta\text{NDVI}$, $\text{dNBR}$, $\Delta\text{BSI}$), Otsu thresholding, and 8-connected change patch statistics.
4. **64-Band AlphaEarth Foundation Embedding Change & Swath Cross-Section (`surface_change`, `cross_section`, `viewer_3d`)**: Load 64-band unit-sphere embeddings (`A00..A63`), compute pixel-wise cosine distance ($1 - u \cdot v$), cosine similarity, and $L_2$ distance in Rust, rank the top changed latent dimensions, and slice a multi-layer topographic & surface-change swath profile across the landslide headscarp and deposit lobe.

In [1]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

import pygeomapper as pgm
ROOT = Path.cwd() if (Path.cwd() / "example_data").exists() else Path.cwd().parent   # repo root (run from notebooks/ or the root)

print(f"pyGeoMapper version: {pgm.__version__} | Rust core active: {pgm.HAS_RUST_CORE}")
DATA_DIR = ROOT / "example_data"
IMG_DIR = ROOT / "image"
IMG_DIR.mkdir(exist_ok=True)


pyGeoMapper version: 0.1.0 | Rust core active: True


## 1. Load Pre/Post DEMs & Sentinel-2 Cubes (`raster_io`) and Compute Terrain Morphometry (`terrain_analysis`)

### 1. What This Step Does & Why We Use This Method
- Loads co-registered pre-event (`Copernicus GLO-30`) and post-event (`USGS 3DEP` LiDAR) Digital Elevation Models alongside bi-temporal Copernicus Sentinel-2 L2A reflectance stacks over the **2014 Oso Landslide, WA** (`EPSG:32610`).
- Computes first- and second-order terrain derivatives in Rust (`compute_terrain_attributes`).

### 2. Governing Equations & Physical Meaning
- **Horn (1981) Slope Angle (`slope_deg`) & Aspect (`aspect_deg`):**
  $$\theta = \arctan\!\left(\sqrt{\left(\frac{\partial z}{\partial x}\right)^2 + \left(\frac{\partial z}{\partial y}\right)^2}\right), \qquad \alpha = \operatorname{atan2}\!\left(-\frac{\partial z}{\partial x},\; \frac{\partial z}{\partial y}\right)$$
- **Topographic Position Index (`TPI`):**
  $$\text{TPI}(x,y) = z(x,y) - \bar{z}_{W_r}(x,y)$$
  where $\bar{z}_{W_r}$ is the mean elevation inside a circular neighborhood of radius $r$ ($>0$ on ridges/scarps, $<0$ in channels/hollows).


In [2]:
dem_pre = pgm.raster_io.load_geotiff(DATA_DIR / "oso_dem_pre_cop30.tif", units="m", name="DEM_Pre_2013")
dem_post = pgm.raster_io.load_geotiff(DATA_DIR / "oso_dem_post_3dep.tif", units="m", name="DEM_Post_3DEP")

s2_2017 = pgm.raster_io.load_multiband_geotiff(DATA_DIR / "oso_s2_20170826.tif", units="reflectance", name="S2_2017")
s2_2024 = pgm.raster_io.load_multiband_geotiff(DATA_DIR / "oso_s2_20240906.tif", units="reflectance", name="S2_2024")

print(f"CRS: {dem_post.crs} | Grid shape: {dem_post.shape} | Pixel spacing: dx={dem_post.dx:.2f} m, dy={dem_post.dy:.2f} m")
print(f"Bounds (UTM m): {tuple(round(b, 1) for b in dem_post.bounds)} | Cell area: {dem_post.cell_area_m2:.2f} m^2")
print(f"Sentinel-2 bands ({s2_2017.n_bands}): {s2_2017.band_names}")

# Compute Rust terrain morphometry on pre- and post-landslide DEMs
hs_pre = pgm.terrain_analysis.compute_hillshade(dem_pre, multi_azimuth=True)
hs_post = pgm.terrain_analysis.compute_hillshade(dem_post, multi_azimuth=True)
slope_post, aspect_post = pgm.terrain_analysis.compute_slope_aspect(dem_post)
curv_post = pgm.terrain_analysis.compute_curvature(dem_post)
rough_post = pgm.terrain_analysis.compute_roughness_and_tpi(dem_post, radius_px=3)

print(f"Post-event slope mean: {np.nanmean(slope_post.data):.2f} deg, max: {np.nanmax(slope_post.data):.2f} deg")
print(f"Post-event 30 m roughness std mean: {np.nanmean(rough_post['roughness'].data):.2f} m")

CRS: EPSG:32610 | Grid shape: (200, 200) | Pixel spacing: dx=11.26 m, dy=-11.26 m
Bounds (UTM m): (584419.1, 5346716.1, 586671.1, 5348968.1) | Cell area: 126.79 m^2
Sentinel-2 bands (6): ['blue', 'green', 'red', 'nir', 'swir1', 'swir2']
Post-event slope mean: 10.41 deg, max: 54.09 deg
Post-event 30 m roughness std mean: 3.27 m


In [3]:
def make_rgb(stack: pgm.RasterStack, stretch: float = 3.8) -> np.ndarray:
    r = stack.band("red").data
    g = stack.band("green").data
    b = stack.band("blue").data
    rgb = np.stack([r, g, b], axis=-1)
    return np.clip(np.power(np.clip(rgb * stretch, 0.0, 1.0), 0.85), 0.0, 1.0)

fig, axes = plt.subplots(2, 2, figsize=(12, 10), constrained_layout=True)
ext = dem_post.extent

im0 = axes[0, 0].imshow(pgm.viewer_3d.drape_overlay_on_hillshade(hs_pre, dem_pre, cmap="terrain", vmin=60, vmax=340), extent=ext)
axes[0, 0].set_title("a) Pre-Landslide DEM (Copernicus GLO-30, 2011–2013)", fontweight="bold")

im1 = axes[0, 1].imshow(pgm.viewer_3d.drape_overlay_on_hillshade(hs_post, dem_post, cmap="terrain", vmin=60, vmax=340), extent=ext)
axes[0, 1].set_title("b) Post-Landslide DEM (USGS 3DEP LiDAR)", fontweight="bold")

axes[1, 0].imshow(make_rgb(s2_2017), extent=ext)
axes[1, 0].set_title("c) Sentinel-2 L2A True Color (2017-08-26)", fontweight="bold")

axes[1, 1].imshow(make_rgb(s2_2024), extent=ext)
axes[1, 1].set_title("d) Sentinel-2 L2A True Color (2024-09-06)", fontweight="bold")

for ax in axes.ravel():
    ax.set_xlabel("UTM Easting (m, EPSG:32610)")
    ax.set_ylabel("UTM Northing (m)")

fig.savefig(IMG_DIR / "terrain_and_rgb.png", dpi=160)
plt.show()

## 2. Multi-Band Optical & Mineral Spectral Indices (`spectral_indices`)

### 1. What This Step Does & Why We Use This Method
- Computes normalized difference and band-ratio indices in Rust that isolate **live vegetation chlorophyll (`NDVI`)**, **canopy moisture/burn state (`NBR`, `NDMI`)**, **exposed bare soil/debris (`BSI`)**, and **hydrothermal clay/iron mineralogy (`ClayRatio`, `FerricOxideRatio`)**.

### 2. Governing Equations & Physical Meaning
$$\text{NDVI} = \frac{\rho_{\text{NIR}} - \rho_{\text{Red}}}{\rho_{\text{NIR}} + \rho_{\text{Red}}}, \qquad \text{NBR} = \frac{\rho_{\text{NIR}} - \rho_{\text{SWIR2}}}{\rho_{\text{NIR}} + \rho_{\text{SWIR2}}}, \qquad \text{BSI} = \frac{(\rho_{\text{SWIR1}} + \rho_{\text{Red}}) - (\rho_{\text{NIR}} + \rho_{\text{Blue}})}{(\rho_{\text{SWIR1}} + \rho_{\text{Red}}) + (\rho_{\text{NIR}} + \rho_{\text{Blue}})}$$
$$\text{ClayRatio} = \frac{\rho_{\text{SWIR1}}}{\rho_{\text{SWIR2}}}, \qquad \text{FerricOxideRatio} = \frac{\rho_{\text{Red}}}{\rho_{\text{Blue}}}$$


In [4]:
indices_2017 = pgm.spectral_indices.compute_index_suite(s2_2017)
indices_2024 = pgm.spectral_indices.compute_index_suite(s2_2024)

for name in ["NDVI", "NBR", "NDWI", "BSI", "ClayRatio", "FerrousIronIndex"]:
    g17 = indices_2017[name].data
    g24 = indices_2024[name].data
    print(f"{name:16s} | 2017 mean={np.nanmean(g17):+.3f} (p5={np.nanpercentile(g17,5):+.3f}, p95={np.nanpercentile(g17,95):+.3f}) "
          f"| 2024 mean={np.nanmean(g24):+.3f} (p5={np.nanpercentile(g24,5):+.3f}, p95={np.nanpercentile(g24,95):+.3f})")

fig, axes = plt.subplots(2, 3, figsize=(14, 8.5), constrained_layout=True)
plot_specs = [
    (indices_2017["NDVI"], "RdYlGn", -0.1, 0.9, "2017 NDVI (Vegetation)"),
    (indices_2017["BSI"], "BrBG_r", -0.5, 0.3, "2017 BSI (Bare Soil / Landslide Scar)"),
    (indices_2017["ClayRatio"], "magma", 0.9, 2.0, "2017 Clay Ratio (SWIR1 / SWIR2)"),
    (indices_2024["NDVI"], "RdYlGn", -0.1, 0.9, "2024 NDVI (Revegetation & Harvests)"),
    (indices_2024["BSI"], "BrBG_r", -0.5, 0.3, "2024 BSI (Bare Soil Index)"),
    (indices_2024["ClayRatio"], "magma", 0.9, 2.0, "2024 Clay Ratio (SWIR1 / SWIR2)"),
]
for ax, (grid, cmap, vmin, vmax, title) in zip(axes.ravel(), plot_specs):
    im = ax.imshow(grid.data, cmap=cmap, vmin=vmin, vmax=vmax, extent=ext)
    ax.set_title(title, fontweight="bold")
    fig.colorbar(im, ax=ax, shrink=0.8)

fig.savefig(IMG_DIR / "spectral_indices.png", dpi=160)
plt.show()

NDVI             | 2017 mean=+0.625 (p5=-0.210, p95=+0.907) | 2024 mean=+0.542 (p5=-0.029, p95=+0.756)
NBR              | 2017 mean=+0.371 (p5=-0.348, p95=+0.700) | 2024 mean=+0.429 (p5=-0.126, p95=+0.699)
NDWI             | 2017 mean=-0.535 (p5=-0.804, p95=+0.182) | 2024 mean=-0.439 (p5=-0.654, p95=+0.053)
BSI              | 2017 mean=-0.441 (p5=-0.842, p95=+0.167) | 2024 mean=-0.473 (p5=-0.734, p95=-0.030)
ClayRatio        | 2017 mean=+0.684 (p5=+0.027, p95=+1.162) | 2024 mean=+0.512 (p5=+0.026, p95=+1.050)
FerrousIronIndex | 2017 mean=+2.189 (p5=+1.669, p95=+2.622) | 2024 mean=+1.635 (p5=+1.406, p95=+1.889)


## 3. Bi-Temporal Surface Change: DEM of Difference (`DoD`) & Spectral Change (`surface_change`)

### 1. What This Step Does & Why We Use This Method
- Differences the post-event and pre-event DEMs (`DoD`) with optional stable-ground planar bias removal and Nuth & Kääb (2011) horizontal shift estimation, then integrates the volumetric mass budget ($V_{\text{eroded}}, V_{\text{deposited}}, V_{\text{net}}$).

### 2. Governing Equations & Physical Meaning
$$\text{DoD}(x,y) = z_{\text{post}}(x,y) - z_{\text{pre}}(x,y) - b_{\text{planar}}(x,y)$$
$$V_{\text{eroded}} = \sum_{\text{DoD} < -\tau_z} |\text{DoD}|\,\Delta x\,\Delta y, \qquad V_{\text{deposited}} = \sum_{\text{DoD} > +\tau_z} \text{DoD}\,\Delta x\,\Delta y$$

### 3. How to Read the Visual Output
- **Blue Zones (`DoD < 0`):** Topographic lowering / evacuation inside the 2014 Oso landslide source headscarp.
- **Red Zones (`DoD > 0`):** Topographic aggradation / debris-avalanche runout lobe across the North Fork Stillaguamish floodplain.


In [5]:
# 1. Compute raw DEM of Difference and identify stable terrain (|raw_dod| < 4.0 m) for co-registration check
raw_dod_res = pgm.surface_change.compute_dod(dem_pre, dem_post)
stable_mask = (np.abs(raw_dod_res["raw_dod"].data) < 4.0).astype(np.uint8)

dod_res = pgm.surface_change.compute_dod(dem_pre, dem_post, detrend_stable_mask=stable_mask)
dod_grid = dod_res["dod"]

# Integrate volumetric erosion & deposition budget above LoD = 2.0 m
vol_budget = pgm.surface_change.compute_volume_budget(dod_grid, lod_threshold=2.0, sigma_dz=0.5)

print("=== Oso Landslide DEM of Difference (DoD) Volume Budget (LoD = 2.0 m) ===")
print(f"Planar bias coeffs removed (c0, cx, cy): {dod_res['planar_bias_coeffs']}")
print(f"Nuth & Kääb residual shift on stable terrain (dx, dy, dz m): {tuple(round(v, 3) for v in dod_res['nuth_kaab_shift_m'])}")
print(f"Eroded area:    {vol_budget['eroded_area_m2']:,.0f} m^2 | Eroded volume:    {vol_budget['eroded_volume_m3']:,.0f} m^3 (max depth: {vol_budget['max_erosion_m']:.1f} m)")
print(f"Deposited area: {vol_budget['deposited_area_m2']:,.0f} m^2 | Deposited volume: {vol_budget['deposited_volume_m3']:,.0f} m^3 (max thickness: {vol_budget['max_deposition_m']:.1f} m)")
print(f"Net volume difference: {vol_budget['net_volume_m3']:+,.0f} ± {vol_budget['volume_uncertainty_m3']:,.0f} m^3")

# 2. Compute bi-temporal spectral changes (dNDVI, dNBR, dBSI) and connected disturbance patches
dndvi = pgm.surface_change.compute_index_change(indices_2017["NDVI"], indices_2024["NDVI"], convention="t2_minus_t1", name="dNDVI")
dnbr = pgm.surface_change.compute_index_change(indices_2017["NBR"], indices_2024["NBR"], convention="t1_minus_t2", name="dNBR")

dnbr_otsu = pgm.surface_change.otsu_threshold(np.abs(dnbr.data))
ero_labels, ero_patches = pgm.surface_change.extract_change_patches(-dod_grid.data, threshold=8.0, min_pixels=15)
print(f"|dNBR| Otsu threshold: {dnbr_otsu:.3f} | Major headscarp erosion patches (>8 m depth): {len(ero_patches)}")
if ero_patches:
    print(f"Largest erosion patch: {ero_patches[0]['pixel_count']} px ({ero_patches[0]['area_m2']:,.0f} m^2), mean erosion={ero_patches[0]['mean_magnitude']:.1f} m")

=== Oso Landslide DEM of Difference (DoD) Volume Budget (LoD = 2.0 m) ===
Planar bias coeffs removed (c0, cx, cy): (-0.2568686018510497, 0.00011958314265024848, -4.795614164716874e-05)
Nuth & Kääb residual shift on stable terrain (dx, dy, dz m): (-0.133, 0.038, 0.013)
Eroded area:    148,468 m^2 | Eroded volume:    1,872,191 m^3 (max depth: 69.4 m)
Deposited area: 926,057 m^2 | Deposited volume: 9,413,331 m^3 (max thickness: 33.4 m)
Net volume difference: +7,541,140 ± 5,836 m^3
|dNBR| Otsu threshold: 0.334 | Major headscarp erosion patches (>8 m depth): 1
Largest erosion patch: 382 px (38,200 m^2), mean erosion=30.3 m


In [6]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.8), constrained_layout=True)

axes[0].imshow(pgm.viewer_3d.drape_overlay_on_hillshade(hs_post, dod_grid, cmap="RdBu", vmin=-45, vmax=45, alpha=0.75), extent=ext)
im_dod = axes[0].imshow(dod_grid.data, cmap="RdBu", vmin=-45, vmax=45, alpha=0.0, extent=ext)
axes[0].set_title("a) DEM of Difference DoD (m, Red=Erosion, Blue=Deposition)", fontweight="bold")
fig.colorbar(im_dod, ax=axes[0], shrink=0.82, label="Elevation Change Δz (m)")

im_ndvi = axes[1].imshow(dndvi.data, cmap="PiYG", vmin=-0.55, vmax=0.55, extent=ext)
axes[1].set_title("b) ΔNDVI (2024 − 2017: Green=Scar Recovery, Pink=Harvest)", fontweight="bold")
fig.colorbar(im_ndvi, ax=axes[1], shrink=0.82, label="ΔNDVI")

im_nbr = axes[2].imshow(dnbr.data, cmap="RdYlBu_r", vmin=-0.55, vmax=0.55, extent=ext)
axes[2].set_title("c) dNBR (2017 − 2024: Red=Canopy Loss, Blue=Regrowth)", fontweight="bold")
fig.colorbar(im_nbr, ax=axes[2], shrink=0.82, label="dNBR")

for ax in axes:
    ax.set_xlabel("UTM Easting (m)")
    ax.set_ylabel("UTM Northing (m)")

fig.savefig(IMG_DIR / "dod_and_spectral_change.png", dpi=160)
plt.show()

## 4. 64-Band AlphaEarth Foundation Embedding Change, Unsupervised Mapping & Swath Cross-Section (`surface_change`, `auto_mapping`, `cross_section`, `viewer_3d`)

### 1. What This Step Does & Why We Use This Method
- Quantifies multi-sensor surface change in the **64-dimensional AlphaEarth Foundations latent space (`A00..A63`)** using pixel-wise angular cosine distance in Rust, and extracts a topographic/DoD swath profile (`A → A'`) from headscarp to runout toe.

### 2. Governing Equations & Physical Meaning
- **AlphaEarth Dequantization & Cosine Distance:**
  $$u_b = \operatorname{sign}(q_b)\left(\frac{q_b}{127}\right)^2, \qquad d_{\cos}(\mathbf{u}_{t_1}, \mathbf{u}_{t_2}) = 1 - \frac{\mathbf{u}_{t_1} \cdot \mathbf{u}_{t_2}}{\|\mathbf{u}_{t_1}\|_2\,\|\mathbf{u}_{t_2}\|_2} \in [0, 2]$$
  - Values near $0$ indicate stable land cover; values $> 0.15$ highlight structural/land-cover transformation on the unit hypersphere $\mathbb{S}^{63}$.


In [7]:
# Load 64-band quantized unit-sphere embeddings (A00..A63) for 2017 and 2024
ae_2017 = pgm.raster_io.load_multiband_geotiff(
    DATA_DIR / "oso_alphaearth_2017.tif", dequantize_alphaearth=True, name="AlphaEarth_2017"
)
ae_2024 = pgm.raster_io.load_multiband_geotiff(
    DATA_DIR / "oso_alphaearth_2024.tif", dequantize_alphaearth=True, name="AlphaEarth_2024"
)

emb_change = pgm.surface_change.compute_embedding_change(ae_2017, ae_2024, threshold="otsu", min_patch_pixels=12)
cos_dist = emb_change["cosine_distance"]
l2_dist = emb_change["l2_distance"]
pc1_change = emb_change["pc1_change"]

print(f"64-Band Embedding shape: {ae_2017.shape} | Unit L2 norm check: mean={np.nanmean(np.linalg.norm(ae_2017.data, axis=2)):.4f}")
print(f"Cosine distance (1 - u·v) range: [{np.nanmin(cos_dist.data):.4f}, {np.nanmax(cos_dist.data):.4f}], Otsu threshold: {emb_change['otsu_threshold']:.4f}")
print(f"Top 8 changed AlphaEarth latent bands: {emb_change['top_changed_band_names'][:8]}")
print(f"Detected multi-modal embedding change patches: {len(emb_change['patches'])}")

# Compare correlation of 64-D embedding cosine distance with both spectral change (|dNBR|) and geomorphic change (|DoD|)
r_spec = np.corrcoef(cos_dist.data.ravel(), np.abs(dnbr.data).ravel())[0, 1]
r_geom = np.corrcoef(cos_dist.data.ravel(), np.abs(dod_grid.data).ravel())[0, 1]
print(f"Correlation of 64-D Cosine Distance with |dNBR| (spectral): r = {r_spec:.3f}")
print(f"Correlation of 64-D Cosine Distance with |DoD|  (terrain):  r = {r_geom:.3f}")

# Unsupervised Rust k-means++ clustering on the 64-band 2024 embedding cube
clusters = pgm.auto_mapping.cluster_units_kmeans(ae_2024, n_clusters=5, standardize=False, seed=42)
print("Unsupervised 64-D embedding cluster summary:", clusters["unit_summary"])

64-Band Embedding shape: (200, 200, 64) | Unit L2 norm check: mean=1.0000
Cosine distance (1 - u·v) range: [0.0026, 1.4665], Otsu threshold: 0.3514
Top 8 changed AlphaEarth latent bands: ['A20', 'A56', 'A09', 'A42', 'A12', 'A40', 'A19', 'A31']
Detected multi-modal embedding change patches: 14
Correlation of 64-D Cosine Distance with |dNBR| (spectral): r = 0.751
Correlation of 64-D Cosine Distance with |DoD|  (terrain):  r = 0.647
Unsupervised 64-D embedding cluster summary: [{'unit_id': 0, 'pixel_count': 9135, 'area_m2': 1158204.726}, {'unit_id': 1, 'pixel_count': 1672, 'area_m2': 211988.8672}, {'unit_id': 2, 'pixel_count': 12794, 'area_m2': 1622120.5544}, {'unit_id': 3, 'pixel_count': 12388, 'area_m2': 1570644.7888}, {'unit_id': 4, 'pixel_count': 4011, 'area_m2': 508545.0636}]


In [8]:
# Extract a N-S topographic and surface-change swath profile across the Oso landslide headscarp and deposit lobe
xmin, ymin, xmax, ymax = dem_post.bounds
x_transect = xmin + 0.46 * (xmax - xmin)
start_xy = (x_transect, ymax - 120.0)  # North (Whitman Bench headscarp)
end_xy = (x_transect, ymin + 120.0)    # South (Valley deposit lobe & NF Stillaguamish River)

multi_prof = pgm.cross_section.extract_multilayer_profile(
    {
        "DEM_Pre": dem_pre,
        "DEM_Post": dem_post,
        "DoD": dod_grid,
        "dNDVI": dndvi,
        "AlphaEarth_CosDist": cos_dist,
    },
    start_xy=start_xy,
    end_xy=end_xy,
    n_samples=200,
    swath_width_m=160.0,
)

fig = plt.figure(figsize=(15, 9.5), constrained_layout=True)
gs = fig.add_gridspec(2, 3)

ax0 = fig.add_subplot(gs[0, 0])
ax0.imshow(pgm.viewer_3d.drape_overlay_on_hillshade(hs_post, cos_dist, cmap="inferno", vmin=0.0, vmax=0.45, alpha=0.75), extent=ext)
im0 = ax0.imshow(cos_dist.data, cmap="inferno", vmin=0.0, vmax=0.45, alpha=0.0, extent=ext)
ax0.plot([start_xy[0], end_xy[0]], [start_xy[1], end_xy[1]], "c--", lw=2.2, label="Swath Profile A–A'")
ax0.legend(loc="lower left", facecolor="white", framealpha=0.85)
ax0.set_title("a) AlphaEarth 64-D Cosine Distance (2017 vs 2024)", fontweight="bold")
fig.colorbar(im0, ax=ax0, shrink=0.8, label="Cosine Distance (1 − u·v)")

ax1 = fig.add_subplot(gs[0, 1])
im1 = ax1.imshow(pc1_change.data, cmap="coolwarm", vmin=-0.5, vmax=0.5, extent=ext)
ax1.set_title("b) AlphaEarth 1st Principal Change Axis (PC1)", fontweight="bold")
fig.colorbar(im1, ax=ax1, shrink=0.8, label="PC1 Change Projection")

ax2 = fig.add_subplot(gs[0, 2], projection="3d")
pgm.viewer_3d.plot_3d_terrain_drape(
    dem_post,
    overlay=cos_dist,
    cmap="inferno",
    vmin=0.0,
    vmax=0.45,
    vertical_exaggeration=2.2,
    azimuth_deg=-58.0,
    elevation_deg=32.0,
    stride=2,
    ax=ax2,
    title="c) 3D Post-Slide DEM Draped with 64-D Change",
)

ax3 = fig.add_subplot(gs[1, :2])
dist = multi_prof["distance_m"]
z_pre_m = multi_prof["layers"]["DEM_Pre"]["mean"]
z_post_m = multi_prof["layers"]["DEM_Post"]["mean"]
z_post_min = multi_prof["layers"]["DEM_Post"]["min"]
z_post_max = multi_prof["layers"]["DEM_Post"]["max"]
ax3.fill_between(dist, z_post_min, z_post_max, color="steelblue", alpha=0.22, label="Post-Slide Swath Min–Max (160 m)")
ax3.plot(dist, z_pre_m, "k--", lw=2.0, label="Pre-Slide Surface (Copernicus DEM 2011–2013)")
ax3.plot(dist, z_post_m, "b-", lw=2.2, label="Post-Slide Surface (USGS 3DEP LiDAR)")
ax3.fill_between(dist, z_post_m, z_pre_m, where=(z_post_m < z_pre_m), color="crimson", alpha=0.35, label="Headscarp Erosion")
ax3.fill_between(dist, z_pre_m, z_post_m, where=(z_post_m >= z_pre_m), color="teal", alpha=0.35, label="Debris-Lobe Deposition")
ax3.set_xlabel("Distance Along Transect A–A' (m, North to South)")
ax3.set_ylabel("Elevation (m NAVD88)")
ax3.set_title("d) Swath Topographic Cross-Section A–A' Across the 2014 Oso Landslide", fontweight="bold")
ax3.grid(True, alpha=0.3)
ax3.legend(loc="upper right")

ax4 = fig.add_subplot(gs[1, 2])
ax4_twin = ax4.twinx()
ax4.plot(dist, multi_prof["layers"]["AlphaEarth_CosDist"]["mean"], color="darkorange", lw=2.2, label="64-D Cosine Dist")
ax4_twin.plot(dist, multi_prof["layers"]["dNDVI"]["mean"], color="forestgreen", lw=1.8, ls="--", label="ΔNDVI (2024−2017)")
ax4.set_xlabel("Distance Along Transect A–A' (m)")
ax4.set_ylabel("AlphaEarth 64-D Cosine Distance", color="darkorange")
ax4_twin.set_ylabel("Spectral ΔNDVI", color="forestgreen")
ax4.set_title("e) Along-Profile Embedding vs. Spectral Change", fontweight="bold")
ax4.grid(True, alpha=0.3)

fig.savefig(IMG_DIR / "alphaearth_and_profile.png", dpi=160)
plt.show()